# Lesson 04 - Designmönstret Verktygsanvändning

I denna lektion kommer du att lära dig designmönstret **Verktygsanvändning** för AI-agenter med Microsoft Agent Framework (Python). Vi täcker:

- Definiera funktionsverktyg med `@tool`-dekorationen och typade parametrar
- Tillhandahålla verktygsscheman så att modellen vet vad varje verktyg gör
- Styra verktygsexekvering med `approval_mode`
- Returnera **strukturerad output** via Pydantic-modeller och `response_format`

Scenariot är en **resebokningsagent** som kan slå upp destinationer, kontrollera tillgänglighet och hämta flyginformation.


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definiera verktyg med @tool-dekoratorn

`@tool`-dekoratorn förvandlar en vanlig Python-funktion till ett verktyg som en agent kan anropa.  
Viktiga punkter:

- **Docstringen** blir verktygsbeskrivningen som modellen ser.
- **Typannoteringar** (inklusive `Annotated` med beskrivningar) definierar verktygets schema.
- `approval_mode` styr om användaren måste godkänna varje anrop innan det körs.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Skapa en agent med flera verktyg

Skicka alla tre verktyg till klienten så att modellen kan använda vilka av dem som helst för att besvara användarens fråga.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturerad utdata med verktyg

Genom att ställa in `response_format` till en Pydantic-modell tvingas agenten att returnera ett vältypat JSON-objekt istället för fri text. Detta är användbart när efterföljande kod behöver konsumera resultatet programmässigt.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mönster för verktygs-godkännande

Parametern `approval_mode` på `@tool` styr om verktygssamtal kräver mänskligt godkännande innan de körs:

| Läget | Beteende |
|---|---|
| `"never_require"` | Verktyget körs automatiskt — ingen användarbekräftelse krävs. |
| `"always_require"` | Varje samtal måste godkännas av användaren innan det utförs. |

Använd `"always_require"` för verktyg som har bieffekter (t.ex. boka en flygresa, debitera ett kreditkort) så att en människa förblir inblandad.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Sammanfattning

I denna lektion lärde du dig hur man:

1. **Definierar verktyg** med hjälp av `@tool`-dekoreraren med typade parametrar och docstrings som fungerar som verktygsschema.
2. **Sätter ihop flera verktyg** så att agenten kan anropa dem i sekvens för att besvara komplexa frågor.
3. **Returnerar strukturerad output** genom att skicka en Pydantic-modell som `response_format`.
4. **Kontrollerar verktygsapproval** med `approval_mode` för att hålla en människa involverad vid känsliga operationer.

Dessa mönster utgör grunden för att bygga pålitliga, produktionsklara agenter som kan interagera med externa system på ett säkert sätt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
